In [0]:
from pyspark.sql import functions as F



In [0]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================


schema_location = "/Volumes/brewing/config/openbrewery_config_volume/schemas/bronze"

checkpoint_path = "/Volumes/brewing/config/openbrewery_config_volume/checkpoints/bronze"

landing_path = "/Volumes/brewing/landing/openbrewery_landing_volume/"

bronze_table = "/Volumes/brewing/bronze/openbrewery_config_volume/"



In [0]:

raw_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("multiLine", "true")
    .load(landing_path)
)

In [0]:
# ==========================================
# EXPLODE DO ARRAY DATA
# ==========================================

bronze_df = (
    raw_stream_df
    .withColumn("brewery", F.explode("data"))

    .select(
        # Metadata original do JSON
        "metadata",

        # Coluna de rescue
        "_rescued_data",

        # Campos da brewery
        F.col("brewery.id").alias("id"),
        F.col("brewery.name").alias("name"),
        F.col("brewery.brewery_type").alias("brewery_type"),
        F.col("brewery.address_1").alias("address_1"),
        F.col("brewery.address_2").alias("address_2"),
        F.col("brewery.address_3").alias("address_3"),
        F.col("brewery.city").alias("city"),
        F.col("brewery.state").alias("state"),
        F.col("brewery.state_province").alias("state_province"),
        F.col("brewery.postal_code").alias("postal_code"),
        F.col("brewery.country").alias("country"),
        F.col("brewery.longitude").alias("longitude"),
        F.col("brewery.latitude").alias("latitude"),
        F.col("brewery.phone").alias("phone"),
        F.col("brewery.website_url").alias("website_url"),
        F.col("brewery.street").alias("street"),

        # metadata técnico extraído diretamente
        F.col("_metadata.file_name").alias("source_file"),
        F.col("_metadata.file_path").alias("source_path"),
        F.col("_metadata.file_modification_time").alias("file_modification_time"),
        F.col("_metadata.file_size").alias("file_size")
    )
)

In [0]:
# ==========================================
# METADADOS TÉCNICOS
# ==========================================

bronze_df = (
    bronze_df

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "ingestion_date",
        F.to_date(F.current_timestamp())
    )

    .withColumn(
        "source_file",
        F.col("_metadata.file_name")
    )

    .withColumn(
        "source_path",
        F.col("_metadata.file_path")
    )

    .withColumn(
        "file_modification_time",
        F.col("_metadata.file_modification_time")
    )

    .withColumn(
        "file_size",
        F.col("_metadata.file_size")
    )

    .withColumn(
        "execution_id",
        F.date_format(
            F.current_timestamp(),
            "yyyyMMddHHmmss"
        )
    )

    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("id"),
                F.col("name"),
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),
            256
        )
    )
)

In [0]:
# ==========================================
# DEDUPLICAÇÃO
# ==========================================

bronze_df = bronze_df.dropDuplicates(["id"])

In [0]:
# ==========================================
# ESCRITA STREAMING PARA DELTA
# ==========================================

query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .option("path", bronze_table)
    .partitionBy("ingestion_date")
    .trigger(availableNow=True)
    .start()
)


# ==========================================
# EXECUÇÃO
# ==========================================

query.awaitTermination()

print("Carga Bronze finalizada com sucesso.")